# Q8 · 4–10 Hz spike–LFP phase locking

Run `q8_ppc.py` in the `con2phys` environment, then run `q8_ppc_stats.R` before executing this notebook. Set `Q8_OUTPUT_DIR` if you are using a custom results directory.

Both analyses match units to LFP from the same mouse and brain area. The primary analysis uses the median voltage trace across channels. The sensitivity analysis computes PPC per channel and takes the median of those scores for each unit.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if os.environ.get('Q8_OUTPUT_DIR'):
    output = Path(os.environ['Q8_OUTPUT_DIR'])
else:
    cwd = Path.cwd()
    repo = next(path for path in (cwd, *cwd.parents) if (path / 'code/questions/q8_ppc.py').is_file())
    output = repo / 'results/q8'

units = pd.read_csv(output / 'q8_ppc_units.csv')
mice = pd.read_csv(output / 'q8_ppc_stats_mice.csv')
means = pd.read_csv(output / 'q8_ppc_stats_means.csv')
contrasts = pd.read_csv(output / 'q8_ppc_stats_contrasts.csv')
trace = pd.read_csv(output / 'q8_ppc_example_lfp.csv')
spikes = pd.read_csv(output / 'q8_ppc_example_spikes.csv')

areas = [1, 2, 3]
colors = ['#0072B2', '#D55E00', '#00856A']
titles = {'median_trace': 'Median LFP trace', 'median_channel': 'Median channel-wise PPC'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

def save_figure(fig, name):
    fig.savefig(output / f'q8_ppc_{name}.png', dpi=180, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

### Spike-to-phase alignment

An example from the primary analysis. Each dot marks the LFP sample nearest a spike. The two-second window was chosen to show multiple spikes.

In [ ]:
start = trace.time_s.iloc[0]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 4.5), layout='constrained')
for ax, field, label in zip(axes, ['filtered_lfp', 'phase_rad'], ['4–10 Hz LFP', 'Phase (rad)']):
    values = trace[field].to_numpy(copy=True)
    if field == 'phase_rad':
        values[np.flatnonzero(np.abs(np.diff(values)) > np.pi) + 1] = np.nan
    ax.plot(trace.time_s - start, values, color='#333333', linewidth=1)
    ax.scatter(spikes.sample_time_s - start, spikes[field], s=20, color='#D55E00', zorder=3)
    ax.set_ylabel(label)
axes[1].set_yticks([-np.pi, 0, np.pi], ['−π', '0', 'π'])
axes[1].set_xlabel(f'Time since {start:.3f} s in recording (s)')
example = trace.iloc[0]
fig.suptitle(f'Mouse {int(example.mouse_id)} · Area {int(example.brain_area)} · Unit {int(example.cluster_id)}')
save_figure(fig, 'alignment')

### Unit-level PPC

Empirical cumulative distributions show the spread and tails of valid unit scores. The horizontal axis is symmetric logarithmic with a linear region around zero. The mouse-level plots below show the quantities used for inference.

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(9, 3.8), layout='constrained')
for (method, title), ax in zip(titles.items(), axes):
    for area, color in zip(areas, colors):
        x = np.sort(units.loc[units.brain_area.eq(area), f'ppc_{method}'].dropna().to_numpy())
        if x.size:
            ax.step(np.r_[x[0], x], np.arange(x.size + 1) / x.size, where='post',
                    color=color, label=f'Area {area} ({len(x)} units)')
    ax.axvline(0, color='0.7', linewidth=0.8)
    ax.set_xscale('symlog', linthresh=0.001)
    ax.set(xlabel='PPC', title=title, ylim=(0, 1.03))
    ax.legend(loc='lower right', frameon=False, fontsize=8)
axes[0].set_ylabel('Cumulative fraction of units')
save_figure(fig, 'unit_distributions')

### Comparison across mice

Each gray line connects the three observed area means from one mouse. Colored points show marginal means and pointwise 95% confidence intervals from a linear mixed-effects model with a mouse random intercept and area-specific residual variances. The contrast table gives model-estimated differences with Tukey-adjusted confidence intervals and p-values. The R script also saves standalone figures of the model means and contrasts.

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True, figsize=(9, 4.2), layout='constrained')
for (method, title), ax in zip(titles.items(), axes):
    data = mice.loc[mice.method.eq(method) & mice.included_in_stats.astype(str).str.lower().eq('true')]
    for _, row in data.pivot(index='mouse_id', columns='brain_area', values='ppc').iterrows():
        ax.plot(areas, row[areas], color='0.75', alpha=0.65, linewidth=0.8, marker='.', markersize=3)
    summary = means.loc[means.method.eq(method)].set_index('brain_area')
    for area, color in zip(areas, colors):
        row = summary.loc[area]
        interval = [[row.estimate - row.lower_95], [row.upper_95 - row.estimate]]
        ax.errorbar(area, row.estimate, yerr=interval, fmt='o', color=color, capsize=4, zorder=5)
    ax.axhline(0, color='0.55', linewidth=0.8, linestyle=':')
    ax.set(xticks=areas, xlim=(0.65, 3.35), xlabel='Brain area', title=title)
axes[0].set_ylabel('Mean unit PPC per mouse and area')
save_figure(fig, 'paired_mouse_means')
display(means.round(6), contrasts.round(6))